# 07 - Annotation queues feed the dataset

Goal: send the traces the evaluators are unsure about to an annotation queue, let reviewers add the good ones to the golden dataset as one extra step, then validate coverage.

Phase 1 frontline users tag traces and review them in annotation queues. Here you build the queue in code and connect it to the dataset from challenge 6.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import inspect
import json
import os
import sys
import time
import uuid
from importlib.metadata import version

for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client
from opentelemetry import baggage, context, trace
from opentelemetry.sdk.trace import SpanProcessor

import attr_coverage as coverage
import bank_agent
import bank_traffic
import enrich
import enrichment_ref as ref
import lab_curate
import lab_links
import otel_setup

client = Client()
PROJECT = os.environ["LANGSMITH_PROJECT"]
print("project:", PROJECT)

In [ ]:
from lab_curate import reference_score as golden_score
print(f"{ref.ensure_traffic(PROJECT)} root runs available")
dataset = lab_curate.ensure_dataset(client)
queue = lab_curate.ensure_queue(client)
print("dataset:", dataset.name, "| queue:", queue.name)

## Step 1: Score recent traces

This uses the code rubric from challenge 6 (`lab_curate.reference_score`). High scorers are golden candidates and low scorers are clearly bad. The middle is where a human adds the most value.

In [ ]:
from collections import defaultdict

runs = enrich.project_runs(client, PROJECT, hours=24 * 30, limit=2000)
agent_traces = {r.trace_id for r in enrich.agent_runs(runs)}
runs = [r for r in runs if r.trace_id in agent_traces]
by_trace = defaultdict(list)
for r in runs:
    by_trace[r.trace_id].append(r)
scored = [(golden_score(r, by_trace[r.trace_id]), r) for r in runs if r.name == "bank_support_turn"]
print(f"{len(scored)} traces scored")

def review_candidates(scored, low, high, limit):
    band = [pair for pair in scored if low <= pair[0] < high]
    return sorted(band, key=lambda pair: pair[0])[:limit]

## Step 2: Send the uncertain ones to the queue

In [ ]:
picked = review_candidates(scored, low=0.5, high=0.95, limit=8)
already = {str(r.id) for r in client.list_runs_from_annotation_queue(queue.id)}
fresh = [r for _, r in picked if str(r.id) not in already]
if fresh:
    client.add_runs_to_annotation_queue(queue.id, runs=[
        {"run_id": r.id, "session_id": r.session_id, "start_time": r.start_time} for r in fresh])
print(f"{len(picked)} candidates; added {len(fresh)} to the queue")
lab_links.queue_link(lab_curate.QUEUE_NAME, hint="this is what an annotator sees: one trace at a time")

## Step 3: Review in LangSmith, and add to the dataset

Open the queue (link above) as a reviewer. For each trace:

1. Read the input and answer, and add your feedback.
2. If the answer is good enough for the golden set, add it to the dataset **in the same screen**. In the queue's settings, set the default dataset to `agent-golden-...`, then use *Add to dataset* on the run.
   That one click is the only extra step for the annotator.

Reviewing in the UI is the real workflow. The next cell stands in for your reviewers so the notebook can continue: it records `human_verdict` feedback, `approve` for the better half.

In [ ]:
import time
ordered = sorted(picked, key=lambda pair: pair[0], reverse=True)
for i, (s, r) in enumerate(ordered):
    client.create_feedback(r.id, key="human_verdict", value="approve" if i < (len(ordered) + 1) // 2 else "reject",
                           comment="stand-in for a human reviewer", source_info={"source": "workshop"},
                           session_id=r.session_id)
time.sleep(3)
print(f"recorded verdicts for {len(ordered)} queued runs")

## Step 4: Promote approved runs

Write `approved_runs(client, runs)`: return the runs that have a `human_verdict` feedback whose `value` is `"approve"`. Use `client.list_feedback(run_ids=[...], feedback_key=["human_verdict"])`.

In [ ]:
# Reference solution: try writing this yourself first.
def approved_runs(client, runs):
    ok = []
    for r in runs:
        feedback = client.list_feedback(run_ids=[r.id], feedback_key=["human_verdict"])
        if any(f.value == "approve" for f in feedback):
            ok.append(r)
    return ok

In [ ]:
queued = list(client.list_runs_from_annotation_queue(queue.id))
ok = approved_runs(client, queued)
added = lab_curate.add_runs(client, dataset, ok, enrich.answer_of, "annotation_queue", {str(r.id): {"reviewed": True} for r in ok})
print(f"{len(ok)} approved, {added} new examples added from the queue")
lab_links.dataset_link(lab_curate.DATASET_NAME, hint="filter examples by metadata source = annotation_queue")

## Step 5: Validate coverage

The hands-on attributes are the ones marked `2`, `3`, `4`, `6` and `7` in the gap report. The track passes when at least 90% are present in your project.

In [ ]:
report = coverage.gap_report(client, PROJECT, write=True)
present, total, missing = coverage.hands_on_coverage(report)
print(f"hands-on coverage: {present}/{total}")
for name in missing:
    print("  still missing:", name)

## Step 6: The rest of the list

These attributes are not built hands-on. Use the same three patterns: resource attributes for process-wide values, a SpanProcessor for per-request values, and span attributes in code for values only the running code knows.

In [ ]:
with __import__("pandas").option_context("display.max_colwidth", 80, "display.max_rows", 100):
    display(coverage.reference_table(report))

In [ ]:
lab_links.project_link(hint="every trace now carries deployment, identity, request, risk and geography metadata; filter on any of it")

**Result:** a queue with reviewed runs, examples promoted into the dataset, coverage printed, and a reference table.

